# LangGraph: Building Complex Agent Workflows

This notebook introduces **LangGraph**, a framework for building stateful, multi-step agent workflows using graph-based execution.

## Learning Objectives
- Understand graph-based agent architectures
- Build state machines with nodes and edges
- Create conditional routing logic
- Implement complex multi-agent workflows

## Related Theoretical Content
- [Multi-Agent AI](../../notes/04-multi_agent_ai/README.md)
- [Agent Workflows](../../notes/04-multi_agent_ai/03-agent_workflows/README.md)
- [LangGraph Patterns](../../notes/04-multi_agent_ai/04-langgraph/README.md)

## Prerequisites
This notebook is self-contained and can run independently.

## Setup: Import Dependencies

Load required libraries for graph-based agent workflows.

In [ ]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from api_keys import get_api_key

print(" Dependencies imported")

## 1. LangGraph Fundamentals

**Key Concepts:**

1. **State**: Shared data structure passed between nodes
2. **Nodes**: Functions that process and update state
3. **Edges**: Connections defining execution flow
4. **Graph**: Collection of nodes and edges

**Why LangGraph?**
- Complex multi-step workflows
- Conditional branching
- Parallel execution
- State persistence
- Debugging and visualization

## 2. Simple Graph: Single LLM Call

Let's start with the simplest possible graph: one node that calls an LLM.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=get_api_key('groq')
)

# Define state structure
class State(TypedDict):
    """State contains the conversation messages."""
    messages: Annotated[list, add_messages]

# Define node function
def llm_node(state: State) -> State:
    """Call the LLM with current messages."""
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

# Build the graph
graph_builder = StateGraph(State)
graph_builder.add_node('llm_call', llm_node)
graph_builder.add_edge(START, 'llm_call')
graph_builder.add_edge('llm_call', END)

# Compile the graph
simple_graph = graph_builder.compile()

print(" Simple graph created")
print("\nGraph structure:")
print(simple_graph.get_graph().draw_ascii())

## 3. Execute Simple Graph

Run the graph with a user query.

In [ ]:
# Execute the graph
result = simple_graph.invoke({
    "messages": [{"role": "user", "content": "Where is London?"}]
})

print("Result:")
print(result['messages'][-1].content)

## 4. Router Agent: Conditional Execution

A **router agent** analyzes queries and routes them to different processing paths.

**Use Case:** RAG vs Web Search
- Documents in vector DB → Use RAG
- General knowledge → Use web search
- Simple facts → Direct LLM

In [ ]:
# Define enhanced state
class RouterState(TypedDict):
    messages: Annotated[list, add_messages]
    route: str  # Which path to take
    context: str  # Retrieved context (if any)

# Router node: Decide which path to take
def router_node(state: RouterState) -> RouterState:
    """Analyze query and decide routing."""
    user_message = state['messages'][-1]['content'].lower()

    if 'stock' in user_message or 'price' in user_message:
        route = 'stock_lookup'
    elif 'search' in user_message or 'latest' in user_message:
        route = 'web_search'
    else:
        route = 'direct_llm'

    print(f"� Router decision: {route}")
    return {'route': route}

# Stock lookup node
def stock_lookup_node(state: RouterState) -> RouterState:
    """Look up stock prices."""
    print("� Stock lookup path")
    context = "Stock data would be fetched here"
    return {'context': context}

# Web search node
def web_search_node(state: RouterState) -> RouterState:
    """Perform web search."""
    print("� Web search path")
    context = "Web search results would be here"
    return {'context': context}

# Final LLM node
def final_llm_node(state: RouterState) -> RouterState:
    """Generate final response."""
    messages = state['messages'].copy()

    # Add context if available
    if state.get('context'):
        messages.append({
            "role": "system",
            "content": f"Context: {state['context']}"
        })

    response = llm.invoke(messages)
    return {'messages': [response]}

# Conditional edge function
def route_condition(state: RouterState) -> str:
    """Return the next node based on route."""
    return state['route']

print(" Router nodes defined")

## 5. Build Router Graph

Construct a graph with conditional routing.

In [ ]:
# Build router graph
router_builder = StateGraph(RouterState)

# Add nodes
router_builder.add_node('router', router_node)
router_builder.add_node('stock_lookup', stock_lookup_node)
router_builder.add_node('web_search', web_search_node)
router_builder.add_node('direct_llm', final_llm_node)
router_builder.add_node('final_response', final_llm_node)

# Add edges
router_builder.add_edge(START, 'router')

# Conditional edges from router
router_builder.add_conditional_edges(
    'router',
    route_condition,
    {
        'stock_lookup': 'stock_lookup',
        'web_search': 'web_search',
        'direct_llm': 'direct_llm'
    }
)

# All paths lead to final response or end
router_builder.add_edge('stock_lookup', 'final_response')
router_builder.add_edge('web_search', 'final_response')
router_builder.add_edge('direct_llm', END)
router_builder.add_edge('final_response', END)

# Compile
router_graph = router_builder.compile()

print(" Router graph created")
print("\nGraph structure:")
print(router_graph.get_graph().draw_ascii())

## 6. Test Router Graph

Execute the graph with different query types.

In [ ]:
# Test 1: Stock query
print("Test 1: Stock query\n" + "="*60)
result1 = router_graph.invoke({
    "messages": [{"role": "user", "content": "What's the stock price of Tesla?"}]
})
print(f"\nResponse: {result1['messages'][-1].content}\n")

In [ ]:
# Test 2: Web search query
print("Test 2: Web search query\n" + "="*60)
result2 = router_graph.invoke({
    "messages": [{"role": "user", "content": "Search for latest AI news"}]
})
print(f"\nResponse: {result2['messages'][-1].content}\n")

In [ ]:
# Test 3: Direct query
print("Test 3: Direct query\n" + "="*60)
result3 = router_graph.invoke({
    "messages": [{"role": "user", "content": "What is the capital of France?"}]
})
print(f"\nResponse: {result3['messages'][-1].content}\n")

## 7. Understanding the Flow

```
User Query
    ↓
  Router (analyzes query)
    ↓
    ├── stock_lookup → final_response → END
    ├── web_search → final_response → END
    └── direct_llm → END
```

**Benefits:**
- Specialized handling per query type
- Efficient resource usage
- Clear execution paths
- Easy to debug and extend

## 8. Advanced Pattern: Agentic RAG

Comment explaining the pattern (code simplified for brevity).

In [ ]:
# Advanced Agentic RAG Pattern (conceptual)
"""
Router Agent:
    ↓
    ├── Query Vector DB
    │       ↓
    │   Relevance Check
    │       ↓
    │   ├── Relevant → Generate Answer
    │   └── Not Relevant → Web Search → Generate Answer
    │
    └── Direct LLM → Generate Answer

Key Features:
- Automatic relevance checking
- Fallback to web search
- Tool use when needed
- Memory and state tracking
"""
print("Agentic RAG pattern explained above")

## Key Takeaways

1. **State Management**: TypedDict defines shared state structure
2. **Nodes**: Functions that process and update state
3. **Edges**: Define execution flow (sequential or conditional)
4. **Conditional Routing**: Dynamic path selection based on state
5. **Visualization**: ASCII diagrams help understand flow

## LangGraph Advantages

- **Explainability**: Clear execution paths
- **Debugging**: Inspect state at each node
- **Modularity**: Reusable nodes
- **Flexibility**: Complex branching logic
- **State Persistence**: Built-in checkpointing

## Common Patterns

1. **Router**: Route to specialized handlers
2. **Retrieval-Augmented**: Fetch context → Generate
3. **Multi-Agent**: Multiple specialized agents collaborate
4. **Human-in-the-Loop**: Pause for human approval
5. **Self-Correction**: Retry with improved prompts

## Production Considerations

- Add error handling nodes
- Implement timeout mechanisms
- Log state transitions
- Monitor execution metrics
- Use persistent checkpointers

## Conclusion

You've now learned:
1. **LLM Basics**: Multiple providers, prompting techniques
2. **Structured Output**: Pydantic schemas for type-safe extraction
3. **RAG**: Basic and advanced retrieval systems
4. **Agentic AI**: Tools and autonomous decision-making
5. **Memory**: In-memory and persistent conversation history
6. **LangGraph**: Complex workflows with state machines

These notebooks provide a foundation for building production-grade AI systems!